# Pipeline correto de treino/teste sem "contaminação" dos dados — K-NN

Este notebook simula, passo a passo, a abertura de um CSV real e o fluxo
**correto** de preparação de dados para treinar um K-NN, evitando dois
problemas que discutimos:

1. **Avaliação otimista** — treinar e testar com os mesmos dados (o modelo
   "decora" em vez de generalizar).
2. **Contaminação/vazamento de dados (*data leakage*)** — calcular
   estatísticas de limpeza (média, desvio-padrão, limites de outlier) usando
   também os dados de teste, deixando informação do teste "vazar" para o
   treino.

Regra de ouro seguida no notebook inteiro:

> **Divide primeiro. Depois só "aprende" (fit) qualquer coisa usando o
> treino. O teste só recebe a transformação já aprendida (transform).**

Base usada: `clientes_knn.csv` (dataset de clientes com o atributo alvo
`segmento`: baixo / médio / alto).


## 1. Importações

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Fixamos uma semente (random_state) em todas as etapas aleatórias do notebook.
# Por quê: garante que o split e os resultados sejam reproduzíveis -- se você
# rodar de novo, obtém exatamente a mesma divisão e os mesmos números.
RANDOM_STATE = 42


## 2. Abrir o CSV (como se fosse um arquivo real no seu computador)

Aqui simulamos o passo que você faria normalmente: abrir o arquivo com
`pandas.read_csv`. Troque o caminho pelo local real do seu `.csv` quando for
usar com seus próprios dados.


In [2]:
caminho_csv = "clientes_knn.csv"
df = pd.read_csv(caminho_csv)
# Por quê: read_csv já entrega o arquivo como DataFrame, pronto para inspecionar.

print("Formato do dataset (linhas, colunas):", df.shape)
df.head()
# Por quê: olhar as primeiras linhas ajuda a confirmar que o arquivo foi lido
# corretamente (colunas certas, separador certo, tipos plausíveis).


Formato do dataset (linhas, colunas): (180, 8)


,id_cliente,idade,renda_mensal,compras_mes,ticket_medio,visitas_site_mes,dias_desde_ultima_compra,segmento
0,1,18,2861,3,79.06,3,104,baixo
1,2,28,1225,3,85.52,2,67,baixo
2,3,34,2166,2,73.12,4,86,baixo
3,4,21,3158,3,76.93,0,87,baixo
4,5,26,2494,3,87.48,3,57,baixo


## 3. Simular sujeira nos dados (só para fins didáticos)

O `clientes_knn.csv` original já está limpo. Para o exemplo ficar realista
(a maioria dos dados do mundo real tem problemas), vamos introduzir de
propósito alguns valores ausentes e um outlier bem exagerado.

**Isso é só para a demonstração** -- com seus dados reais você pularia esta
célula e usaria o `df` já carregado.


In [3]:
rng = np.random.default_rng(RANDOM_STATE)
df_sujo = df.copy()
# Por quê: trabalhamos numa cópia para não perder o dataset original limpo.

indices_nan = rng.choice(df_sujo.index, size=8, replace=False)
df_sujo.loc[indices_nan, "renda_mensal"] = np.nan
# Por quê: simula sensores/formulários que às vezes não capturam o dado --
# um problema comum em bases reais (valores ausentes).

indice_outlier = rng.choice(df_sujo.index)
df_sujo.loc[indice_outlier, "renda_mensal"] = 500000
# Por quê: simula um erro de digitação/medição -- um valor absurdamente alto
# de renda mensal para testar nosso tratamento de outliers mais adiante.

df = df_sujo
print("Valores ausentes introduzidos:", df['renda_mensal'].isna().sum())


Valores ausentes introduzidos: 8


## 4. Inspeção geral ANTES da divisão

Aqui só **olhamos** os dados -- não calculamos nada que será usado para
preencher, cortar ou escalar valores. Inspecionar (ver que existem nulos,
ver tipos de coluna) não contamina nada; o problema começaria se
calculássemos, por exemplo, a média para *imputação* usando a base inteira.


In [4]:
df.info()
# Por quê: confirma tipos de cada coluna e mostra rapidamente quais colunas
# têm valores não nulos (contagem "non-null").

df.isnull().sum()
# Por quê: contagem explícita de valores ausentes por coluna -- ajuda a
# decidir a estratégia de tratamento mais adiante (aqui: imputação).

df.describe()
# Por quê: dá uma ideia geral de escala e possíveis outliers (olhando
# min/max/quartis) -- mas note que ainda NÃO estamos usando esses números
# para tratar nada, é só diagnóstico visual.


<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_cliente                180 non-null    int64  
 1   idade                     180 non-null    int64  
 2   renda_mensal              172 non-null    float64
 3   compras_mes               180 non-null    int64  
 4   ticket_medio              180 non-null    float64
 5   visitas_site_mes          180 non-null    int64  
 6   dias_desde_ultima_compra  180 non-null    int64  
 7   segmento                  180 non-null    str    
dtypes: float64(2), int64(5), str(1)
memory usage: 11.4 KB


,id_cliente,idade,renda_mensal,compras_mes,ticket_medio,visitas_site_mes,dias_desde_ultima_compra
count,180.000000,180.000000,172.000000,180.000000,180.000000,180.000000,180.000000
mean,90.500000,36.772222,9458.325581,5.338889,223.202000,5.505556,41.127778
std,52.105662,11.506049,37852.734188,3.173148,155.453739,3.241658,31.029096
min,1.000000,18.000000,1200.000000,0.000000,36.900000,0.000000,1.000000
25%,45.750000,26.750000,2957.000000,3.000000,85.117500,3.000000,15.000000
50%,90.500000,37.000000,5652.000000,5.000000,181.355000,5.000000,34.000000
75%,135.250000,46.000000,9061.000000,8.000000,345.292500,7.250000,57.250000
max,180.000000,65.000000,500000.000000,13.000000,676.900000,17.000000,120.000000


## 5. Dividir treino e teste (70/30) ANTES de qualquer tratamento

Este é o passo-chave: a divisão acontece **antes** de imputar, tratar
outliers ou padronizar. Assim, nenhuma estatística usada na limpeza é
calculada olhando para o teste.


In [5]:
colunas_features = [
    "idade",
    "renda_mensal",
    "compras_mes",
    "ticket_medio",
    "visitas_site_mes",
    "dias_desde_ultima_compra",
]
X = df[colunas_features]
y = df["segmento"]
# Por quê: separamos atributos (X) do alvo (y). "id_cliente" fica de fora
# por ser apenas um identificador, sem valor preditivo.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)
# Por quê test_size=0.30: reserva 30% dos dados só para avaliação final,
# nunca usados no treino -- é o que garante uma medida honesta de
# generalização.
# Por quê stratify=y: mantém a mesma proporção das classes (baixo/médio/alto)
# em treino e teste, evitando um split desbalanceado por acaso.
# Por quê random_state: reprodutibilidade -- mesmo split sempre que rodar.

print("Treino:", X_train.shape, " Teste:", X_test.shape)


Treino: (126, 6)  Teste: (54, 6)


## 6. Tratar valores ausentes -- aprender (fit) só no treino

A partir daqui, toda estatística de limpeza é **aprendida apenas com o
treino** e depois **aplicada** (sem reaprender) no teste.


In [6]:
imputer = SimpleImputer(strategy="median")
# Por quê median: menos sensível a outliers do que a média -- e ainda temos
# aquele outlier de renda_mensal = 500000 no treino ou no teste.
# pega todos os valores NaN

#Especificando estrategias para cada tipo de coluna
'''
colunas_num = ["idade", "renda", "tempo_emprego", "score_credito"]
colunas_cat = ["cidade"]

prep = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                       ("scaler", StandardScaler())]), colunas_num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), colunas_cat),
])

X = df.drop(columns="aprovado")
X_pronto = prep.fit_transform(X)
'''

# Todas as colunas de um dataframe
X_train_imputado = pd.DataFrame(
    imputer.fit_transform(X_train),  #Cuidado aqui o algoritmo esta aprendendo
    columns=colunas_features,        #aprende parametros media, maxima, categoria
    index=X_train.index,
)
# Por quê fit_transform no treino: o imputer CALCULA a mediana de cada
# coluna usando só os dados de treino (fit) e já preenche os ausentes do
# próprio treino (transform) nesse mesmo passo.

X_test_imputado = pd.DataFrame(
    imputer.transform(X_test),     #Aqui esta apenas aplicando o que aprendeu
    columns=colunas_features,
    index=X_test.index,
)
# Por quê apenas transform no teste (sem fit): usamos a MESMA mediana
# aprendida no treino para preencher os ausentes do teste. Se recalculássemos
# a mediana com o teste, seria vazamento de informação do teste para a etapa
# de preparação dos dados.


## 7. Tratar outliers -- limites calculados só com o treino

Vamos usar a regra do IQR (intervalo interquartil): qualquer valor fora de
`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` é considerado outlier. Os limites são
calculados **só com o treino** e depois aplicados (clip) em treino e teste.


In [7]:
def calcular_limites_iqr(serie_treino):
    q1 = serie_treino.quantile(0.25)
    q3 = serie_treino.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return limite_inferior, limite_superior
# Por quê definir como função: deixa claro que a lógica depende só da
# "serie_treino" recebida -- reforça que os limites vêm exclusivamente do
# treino, nunca do teste.

limite_inf, limite_sup = calcular_limites_iqr(X_train_imputado["renda_mensal"])
print(f"Limites de renda_mensal (aprendidos no treino): [{limite_inf:.2f}, {limite_sup:.2f}]")

X_train_tratado = X_train_imputado.copy()
X_test_tratado = X_test_imputado.copy()

X_train_tratado["renda_mensal"] = X_train_tratado["renda_mensal"].clip(limite_inf, limite_sup)
X_test_tratado["renda_mensal"] = X_test_tratado["renda_mensal"].clip(limite_inf, limite_sup)
# Por quê clip (não drop) no teste: no teste, em geral, não descartamos
# linhas -- ele deve continuar representando o mundo real que o modelo vai
# enfrentar em produção. Usamos os MESMOS limites do treino para não deixar
# o teste "ditar" seu próprio critério de corte.


Limites de renda_mensal (aprendidos no treino): [-5983.12, 18053.88]


## 8. Padronizar os atributos -- fit no treino, transform no teste

O K-NN é baseado em distância, então atributos em escalas muito diferentes
(ex.: `idade` de 18-70 vs. `renda_mensal` na casa dos milhares) dominariam o
cálculo se não forem padronizados.


In [8]:
scaler = StandardScaler()
# Por quê StandardScaler: transforma cada atributo para média 0 e desvio
# padrão 1, colocando todos os atributos na mesma escala para o cálculo de
# distância do K-NN.

X_train_final = scaler.fit_transform(X_train_tratado)
# Por quê fit_transform no treino: o scaler aprende média e desvio-padrão de
# cada coluna usando só o treino, e já aplica a transformação nele.

X_test_final = scaler.transform(X_test_tratado)
# Por quê apenas transform no teste: aplica a MESMA média/desvio aprendidos
# no treino. Se fizéssemos fit_transform aqui, o teste teria sua própria
# escala -- e comparar treino e teste em escalas diferentes, além de vazar
# estatísticas do teste, quebraria a lógica do modelo.


## 9. Treinar o K-NN (somente com o treino já tratado)

In [9]:
modelo_knn = KNeighborsClassifier(n_neighbors=5)
# Por quê n_neighbors=5: valor inicial comum/didático -- na prática se testa
# mais de um valor de k com validação cruzada (assunto da aula de 22/09).

modelo_knn.fit(X_train_final, y_train)
# Por quê fit só com X_train_final/y_train: o modelo só pode aprender com os
# dados de treino -- é o que garante que a avaliação no teste seja honesta.


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


## 10. Avaliar no teste (dados que o modelo NUNCA viu no treino)

In [10]:
y_pred = modelo_knn.predict(X_test_final)
# Por quê X_test_final: os mesmos dados de teste, já tratados e escalados
# com as regras aprendidas no treino -- nunca dados "crus" nem retratados do
# zero.

acuracia = accuracy_score(y_test, y_pred)
print(f"Acurácia no teste (pipeline correto): {acuracia:.2%}")

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred))
# Por quê essas três métricas: acurácia dá uma visão geral, mas o
# classification_report (precisão/recall/F1 por classe) e a matriz de
# confusão mostram ONDE o modelo erra -- essencial quando há mais de duas
# classes, como aqui (baixo/médio/alto).


Acurácia no teste (pipeline correto): 100.00%

Relatório de classificação:
              precision    recall  f1-score   support

        alto       1.00      1.00      1.00        18
       baixo       1.00      1.00      1.00        18
       medio       1.00      1.00      1.00        18

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Matriz de confusão:
[[18  0  0]
 [ 0 18  0]
 [ 0  0 18]]


## 11. Bônus (comparação): o jeito ERRADO -- contaminação dos dados

Só para deixar visível o efeito do problema que discutimos: e se
treinássemos e avaliássemos com a base **inteira**, sem separar treino e
teste?


In [11]:
scaler_errado = StandardScaler()
X_todos_escalado = scaler_errado.fit_transform(df[colunas_features].fillna(df[colunas_features].median()))
# Por quê está errado: o scaler aprende média/desvio usando TODOS os dados,
# inclusive os que deveriam ser "teste" -- e ainda preenchemos ausentes com
# a mediana da base inteira. Informação vaza para o "treino" inteiro.

modelo_errado = KNeighborsClassifier(n_neighbors=5)
modelo_errado.fit(X_todos_escalado, df["segmento"])
# Por quê está errado: treinamos com 100% dos dados.

y_pred_errado = modelo_errado.predict(X_todos_escalado)
# Por quê está errado: avaliamos nos MESMOS dados usados no treino -- no
# K-NN, cada ponto encontra a si mesmo como vizinho mais próximo (distância
# zero), então a acurácia fica artificialmente altíssima.

acuracia_errada = accuracy_score(df["segmento"], y_pred_errado)
print(f"Acurácia 'no teste' (jeito ERRADO, sem split): {acuracia_errada:.2%}")
print(f"Acurácia no teste (pipeline CORRETO, com split): {acuracia:.2%}")


Acurácia 'no teste' (jeito ERRADO, sem split): 98.89%
Acurácia no teste (pipeline CORRETO, com split): 100.00%


### Por que os dois números ficaram próximos aqui

Este dataset foi criado como material didático, então as classes
(`baixo` / `médio` / `alto`) são bem separáveis -- por isso até o pipeline
**correto** chega perto de 100% no teste, e a diferença para o jeito errado
parece pequena.

Isso **não** significa que o jeito errado seja seguro. O problema de fundo
continua: no cenário errado, o modelo está sendo avaliado com dados que ele
já usou no treino (inclusive vendo a si mesmo como vizinho mais próximo no
K-NN), então esse número deixa de significar "capacidade de generalizar" --
ele só mede memorização. Em uma base real, mais ruidosa e com classes menos
separáveis (a maioria dos casos do mundo real), essa mesma prática costuma
inflar a acurácia de forma muito mais dramática, escondendo um modelo que na
prática erraria bastante com dados novos.


## Resumo -- o que este notebook garantiu

1. **Split antes de tudo** (70% treino / 30% teste), com `stratify` para
   manter a proporção das classes.
2. **Imputação de ausentes**: mediana calculada só no treino (`fit` no
   treino, `transform` no teste).
3. **Tratamento de outliers**: limites do IQR calculados só no treino,
   aplicados (clip) em treino e teste.
4. **Padronização**: `StandardScaler` ajustado só no treino, aplicado no
   teste.
5. **Treino do K-NN** apenas com o treino já tratado.
6. **Avaliação** no teste -- dados que o modelo nunca viu -- gerando uma
   estimativa honesta de desempenho.
7. **Comparação final** mostrando como pular a divisão treino/teste (ou
   calcular estatísticas com a base inteira) infla artificialmente a
   acurácia -- exatamente o problema de "contaminação do treino" que
   discutimos.
